# site-mapper workflow

In this notebook we go through the site-mapper workflow which obtains species occurrence GBIF data and maps them on the 'natuurbeheerplannen' data layer of the flemish government to investigate in which natural areas species of interest have recently been observed (past 20 years in example). The outputs are a map masking all areas of interest and .csv file with coordinates of occurrences. 

For it to work, you need to create a GBIF account and set your username/pw in an .env file. Moreover, the base 'natuurbeheerplannen (ps_nbhp.lb72.gpkg)' layer can be fetched here: https://www.vlaanderen.be/datavindplaats/catalogus/natuurbeheerplannen

In [11]:
# load dependencies 
# if anything not installed, install through pip install -r requirements.txt

import os
import sys
import geopandas as gpd
import pandas as pd
import folium
import time
from pygbif import occurrences
from dotenv import load_dotenv
import zipfile
from pygbif import species
from branca.element import Template, MacroElement, Element
from folium.plugins import FastMarkerCluster

## 1. Fetch the GBIF bulk data

Make sure to set GBIF credentials in .env file before executing.

In [13]:
# 1. Load credentials from .env
load_dotenv()
gbif_user = os.getenv('GBIF_USER')
gbif_pwd = os.getenv('GBIF_PWD')
gbif_email = os.getenv('GBIF_EMAIL')

In [16]:
# Quick sanity check
assert gbif_user and gbif_pwd and gbif_email, "Missing GBIF credentials in .env"

In [ ]:
# get taxon key for the species in question
result = species.name_backbone(scientificName='Rhinanthus minor')
print(result)

{'usage': {'key': '3172043', 'name': 'Rhinanthus minor L.', 'canonicalName': 'Rhinanthus minor', 'authorship': 'L.', 'rank': 'SPECIES', 'status': 'ACCEPTED', 'genericName': 'Rhinanthus', 'specificEpithet': 'minor', 'type': 'SCIENTIFIC', 'formattedName': '<i>Rhinanthus</i> <i>minor</i> L.'}, 'classification': [{'key': '6', 'name': 'Plantae', 'rank': 'KINGDOM'}, {'key': '7707728', 'name': 'Tracheophyta', 'rank': 'PHYLUM'}, {'key': '220', 'name': 'Magnoliopsida', 'rank': 'CLASS'}, {'key': '408', 'name': 'Lamiales', 'rank': 'ORDER'}, {'key': '6651', 'name': 'Orobanchaceae', 'rank': 'FAMILY'}, {'key': '3172042', 'name': 'Rhinanthus', 'rank': 'GENUS'}, {'key': '3172043', 'name': 'Rhinanthus minor', 'rank': 'SPECIES'}], 'diagnostics': {'matchType': 'EXACT', 'confidence': 97, 'timeTaken': 11, 'timings': {'nameNRank': 0, 'sciNameMatch': 11, 'nameParse': 1, 'luceneMatch': 10}}, 'synonym': False}


In [28]:
def get_gbif_predicates(taxon_key, country="BE", year=2006):
    return {
        "type": "and",
        "predicates": [
            {"type": "equals",              "key": "TAXON_KEY", "value": str(taxon_key)},
            {"type": "equals",              "key": "COUNTRY",   "value": country},
            {"type": "equals",              "key": "HAS_COORDINATE", "value": "true"},
            {"type": "greaterThanOrEquals", "key": "YEAR",      "value": str(year)}
        ]
    }


def download_gbif_data(predicates, download_path="Rhinanthus_BE_20years.zip", poll_interval=60, timeout=7200):
    # 1. Check if already downloaded
    if os.path.exists(download_path):
        print(f"File already exists: {download_path}. Skipping.")
        return download_path

    # 2. Submit request
    print("Submitting request to GBIF...")
    res = occurrences.download(predicates, user=gbif_user, pwd=gbif_pwd, email=gbif_email)
    download_key = res[0]  # res is a tuple: (key, payload)
    print(f"Download key: {download_key}")

    # 3. Poll until done
    print("Polling for completion...")
    start_time = time.time()

    while True:
        meta = occurrences.download_meta(download_key)
        status = meta.get('status')
        print(f"  [{time.strftime('%H:%M:%S')}] Status: {status}")

        if status == 'SUCCEEDED':
            print("Job succeeded!")
            break
        elif status in ['FAILED', 'KILLED', 'CANCELLED']:
            raise Exception(f"Job failed with status: {status}")
        elif (time.time() - start_time) > timeout:
            raise Exception("Timed out waiting for GBIF download.")

        time.sleep(poll_interval)

    # 4. Fetch zip
    print("Fetching zip from GBIF...")
    raw_zip = f"{download_key}.zip"
    if not os.path.exists(raw_zip):
        occurrences.download_get(download_key, path='.')

    # 5. Validate
    if not os.path.exists(raw_zip):
        raise FileNotFoundError(f"Zip not found after download: {raw_zip}")

    size = os.path.getsize(raw_zip)
    print(f"Zip size: {size} bytes")
    if size < 1000:
        with open(raw_zip, 'rb') as f:
            print(f"Suspiciously small — raw content: {f.read()}")
        raise ValueError(f"Zip too small ({size} bytes), likely an error response from GBIF.")

    # 6. Rename and inspect
    os.rename(raw_zip, download_path)
    print(f"Saved to: {download_path}")

    with zipfile.ZipFile(download_path) as z:
        print(f"Zip contents: {z.namelist()}")

    return download_path


if __name__ == "__main__":
    taxon_key = 3172043  # Rhinanthus minor
    predicates = get_gbif_predicates(taxon_key)

    try:
        zip_file = download_gbif_data(predicates)
        print(f"Done! Data saved at: {zip_file}")
    except Exception as e:
        print(f"Error: {e}")

Submitting request to GBIF...


INFO:Your download key is 0065985-260519110011954


Download key: 0065985-260519110011954
Polling for completion...
  [15:49:40] Status: PREPARING
  [15:50:40] Status: PREPARING
  [15:51:40] Status: PREPARING
  [15:52:40] Status: PREPARING
  [15:53:40] Status: PREPARING
  [15:54:40] Status: PREPARING
  [15:55:40] Status: PREPARING
  [15:56:40] Status: PREPARING
  [15:57:40] Status: PREPARING
  [15:58:41] Status: PREPARING
  [15:59:41] Status: PREPARING
  [16:00:41] Status: PREPARING
  [16:01:41] Status: PREPARING
  [16:02:41] Status: PREPARING
  [16:03:41] Status: PREPARING
  [16:04:41] Status: PREPARING
  [16:05:41] Status: PREPARING
  [16:06:41] Status: PREPARING
  [16:07:41] Status: PREPARING
  [16:08:41] Status: PREPARING
  [16:09:41] Status: PREPARING
  [16:10:42] Status: PREPARING
  [16:11:42] Status: PREPARING
  [16:12:42] Status: PREPARING
  [16:13:42] Status: PREPARING
  [16:14:42] Status: PREPARING
  [16:15:42] Status: PREPARING
  [16:16:42] Status: PREPARING
  [16:17:42] Status: PREPARING
  [16:18:42] Status: PREPARING
  [16:

INFO:Download file size: 244070 bytes


  [16:27:43] Status: SUCCEEDED
Job succeeded!
Fetching zip from GBIF...


INFO:On disk at ./0065985-260519110011954.zip


Zip size: 244070 bytes
Saved to: Rhinanthus_BE_20years.zip
Zip contents: ['0065985-260519110011954.csv']
Done! Data saved at: Rhinanthus_BE_20years.zip


## 2. Loading in the data

In [15]:
# 1. Load Data
print("Loading data...")
# Loading occurrences as you specified
zip_file = "Rhinanthus_BE_20years.zip"
df_occ = pd.read_csv(zip_file, sep='\t')
gdf_occ = gpd.GeoDataFrame(
    df_occ, 
    geometry=gpd.points_from_xy(df_occ.decimalLongitude, df_occ.decimalLatitude),
    crs="EPSG:4326"
)

# Loading management plans
gdf_nbhp = gpd.read_file("ps_nbhp.lb72.gpkg")
gdf_nbhp = gdf_nbhp.to_crs(epsg=4326)

# After loading gdf_nbhp, add this before clean_for_folium
# tolerance in degrees: 0.0001 ≈ 10m, 0.001 ≈ 100m — try 0.0005 first
gdf_nbhp = gdf_nbhp.copy()
gdf_nbhp['geometry'] = gdf_nbhp.geometry.simplify(tolerance=0.0005, preserve_topology=True)



Loading data...


c:\Users\simon\Documents\GitHub\site-mapper\.venv\Lib\site-packages\pyogrio\geopandas.py:382: UserWarning: More than one layer found in 'ps_nbhp.lb72.gpkg': 'ps_nbhp' (default), 'layer_styles'. Specify layer parameter to avoid this warning.
  result = read_func(


In [33]:
df_occ.head()

,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,...,identifiedBy,dateIdentified,license,rightsHolder,recordedBy,typeStatus,establishmentMeans,lastInterpreted,mediaType,issue
0,930762526,271c444f-f8d8-4986-b748-e7367755c0c1,INBO:FLORA:04333158,Plantae,Tracheophyta,Magnoliopsida,Lamiales,Orobanchaceae,Rhinanthus,Rhinanthus minor,...,NaN,NaN,CC0_1_0,NaN,Luc Menschaert,NaN,NaN,2026-06-11T17:01:41.365Z,NaN,NaN
1,911436623,271c444f-f8d8-4986-b748-e7367755c0c1,INBO:FLORA:04323588,Plantae,Tracheophyta,Magnoliopsida,Lamiales,Orobanchaceae,Rhinanthus,Rhinanthus minor,...,NaN,NaN,CC0_1_0,NaN,Marleen Massonnet,NaN,NaN,2026-06-11T17:00:47.521Z,NaN,NaN
2,895579313,271c444f-f8d8-4986-b748-e7367755c0c1,INBO:FLORA:04313800,Plantae,Tracheophyta,Magnoliopsida,Lamiales,Orobanchaceae,Rhinanthus,Rhinanthus minor,...,NaN,NaN,CC0_1_0,NaN,Wouter Van Landuyt,NaN,NaN,2026-06-11T17:00:46.970Z,NaN,NaN
3,895579219,271c444f-f8d8-4986-b748-e7367755c0c1,INBO:FLORA:04313742,Plantae,Tracheophyta,Magnoliopsida,Lamiales,Orobanchaceae,Rhinanthus,Rhinanthus minor,...,NaN,NaN,CC0_1_0,NaN,Wouter Van Landuyt,NaN,NaN,2026-06-11T17:01:08.527Z,NaN,NaN
4,895347492,271c444f-f8d8-4986-b748-e7367755c0c1,INBO:FLORA:04310623,Plantae,Tracheophyta,Magnoliopsida,Lamiales,Orobanchaceae,Rhinanthus,Rhinanthus minor,...,NaN,NaN,CC0_1_0,NaN,Bart Hoelbeek,NaN,NaN,2026-06-11T17:01:21.516Z,NaN,NaN


In [5]:
gdf_nbhp.head()

,id,naamdossier,registratienummer,natuurbeheerplantype,goedkeuringsdatum,einddatumnatuurbeheerplan,eigendomtype,einddatumerkenningreservaat,opp_ha,inspireid,geometry
0,26668,Vallei van de Krombeek,NBP-LI-20-0005,Natuurbeheerplan Type 4,2021-11-11,2045-11-11,Privaatrechtelijke rechtspersoon,2099-12-31,9.4698,NatuurbeheerplanType4-NBP-LI-20-0005,"MULTIPOLYGON (((5.57141 50.86974, 5.57161 50.8..."
1,39562,Privébossen Meerle,NBP-AN-19-0041,Natuurbeheerplan Type 2,2022-08-05,2045-04-29,Natuurlijke persoon,NaT,7.5259,NatuurbeheerplanType2-NBP-AN-19-0041,"MULTIPOLYGON (((4.8221 51.43236, 4.82121 51.43..."
2,102896,Gemeente Tessenderlo,NBP-LI-22-0117,Natuurbeheerplan Type 2,2024-07-24,2027-07-03,Bestuur,NaT,43.2935,NatuurbeheerplanType2-NBP-LI-22-0117,"MULTIPOLYGON (((5.15194 51.06057, 5.15155 51.0..."
3,1639,Het Broek,GIBP-AN-14-0033,Natuurbeheerplan Type 4,2015-08-03,2023-10-27,Agentschap voor Natuur en Bos,2099-12-31,28.0138,NatuurbeheerplanType4-GIBP-AN-14-0033,"MULTIPOLYGON (((4.38966 51.0538, 4.38918 51.05..."
4,13547,Kapellenbos,NBP-VB-19-0013,Natuurbeheerplan Type 2,2020-08-27,2044-08-27,Natuurlijke persoon,NaT,6.5692,NatuurbeheerplanType2-NBP-VB-19-0013,"MULTIPOLYGON (((4.8788 50.87757, 4.87932 50.87..."


## 3. Join the data and prepare for mapping

In [ ]:
# 2. Processing (Spatial Join)
print("Joining occurrences with Management Plans...")
gdf_occ = gdf_occ.dropna(subset=['decimalLatitude', 'decimalLongitude'])
joined = gpd.sjoin(gdf_nbhp, gdf_occ, how="inner", predicate="intersects")
highlighted_plans_map = joined[~joined.index.duplicated(keep='first')]

Joining occurrences with Management Plans...


In [ ]:
# 3. Simplify + Prepare for Mapping
gdf_nbhp['geometry'] = gdf_nbhp.geometry.simplify(tolerance=0.0005, preserve_topology=True)
highlighted_plans_map = highlighted_plans_map.copy()
highlighted_plans_map['geometry'] = highlighted_plans_map.geometry.simplify(tolerance=0.0005, preserve_topology=True)

def clean_for_folium(gdf):
    gdf_clean = gdf.copy()
    for col in gdf_clean.columns:
        if col != 'geometry':
            gdf_clean[col] = gdf_clean[col].astype(str).replace(['NaT', 'nan'], 'None')
    return gdf_clean

gdf_nbhp_folium          = clean_for_folium(gdf_nbhp)
highlighted_plans_folium = clean_for_folium(highlighted_plans_map)


## 4. Generate the map

In [ ]:
# 4. Map Visualization
print("Generating Folium map...")
m = folium.Map(location=[51.05, 4.36], zoom_start=8)

title_html = '<h3 align="center" style="font-size:20px"><b>Site selection map for Rhinantus minor</b></h3>'
m.get_root().html.add_child(Element(title_html))

# Add all plans (Grey) — no tooltip, matching working version
folium.GeoJson(
    gdf_nbhp_folium,
    name="All Management Plans (Grey)",
    style_function=lambda x: {'fillColor': 'grey', 'color': 'grey', 'weight': 0.5, 'fillOpacity': 0.2}
).add_to(m)

# Add highlighted plans (Green) — no tooltip
folium.GeoJson(
    highlighted_plans_folium,
    name="Plans with Occurrences (Red)",
    style_function=lambda x: {'fillColor': 'red', 'color': 'red', 'weight': 1, 'fillOpacity': 0.6}
).add_to(m)

# Get only occurrence indices that fall within a management plan
occ_in_plans = gdf_occ[gdf_occ.index.isin(joined['index_right'])]

# Occurrences — individual black dots, only within management plans
occ_group = folium.FeatureGroup(name="Occurrences")
for _, row in occ_in_plans.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color='black',
        fill=True,
        fill_color='black',
        fill_opacity=0.3,
    ).add_to(occ_group)
occ_group.add_to(m)

# Legend
template = """
{% macro html(this, kwargs) %}
<div style="position: fixed; 
     bottom: 50px; left: 50px; width: 250px; height: 110px; 
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white; opacity: 0.85; padding: 10px;">
     <b>Nature site status</b><br>
     <i class="fa fa-square fa-1x" style="color:red"></i> Rhinanthus present<br>
     <i class="fa fa-square fa-1x" style="color:grey"></i> No Rhinanthus detected<br>
     <i class="fa fa-circle fa-1x" style="color:black"></i> Occurrence
</div>
{% endmacro %}
"""
macro = MacroElement()
macro._template = Template(template)
m.add_child(macro)

folium.LayerControl().add_to(m)
m.save("rhinanthus_site_map.html")
print("Map saved.")

Generating Folium map...


## 5. Generate the .csv with details of areas of interest

In [ ]:
# 5. CSV Export (The full set)
print("Exporting tabular overlap data...")
df_export = joined.copy()

# Add Centroid Coordinates
# Instead of computing centroid in EPSG:4326, project first
df_export['plan_centroid_lat'] = df_export.geometry.to_crs(epsg=3857).centroid.to_crs(epsg=4326).y
df_export['plan_centroid_lon'] = df_export.geometry.to_crs(epsg=3857).centroid.to_crs(epsg=4326).x

# Rename occurrence coordinates
df_export = df_export.rename(columns={
    'decimalLatitude': 'occurrence_lat', 
    'decimalLongitude': 'occurrence_lon'
})

# Drop spatial helper columns
cols_to_drop = ['geometry', 'index_right']
df_export = df_export.drop(columns=cols_to_drop, errors='ignore')

# Save to CSV
df_export.to_csv("rhinanthus_overlap_areas.csv", index=False, encoding='utf-8-sig')
print("CSV saved as rhinanthus_overlap_areas.csv")

Exporting tabular overlap data...


C:\Users\simon\AppData\Local\Temp\ipykernel_19268\2044041018.py:6: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  df_export['plan_centroid_lat'] = df_export.geometry.centroid.y
C:\Users\simon\AppData\Local\Temp\ipykernel_19268\2044041018.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  df_export['plan_centroid_lon'] = df_export.geometry.centroid.x


CSV saved as rhinanthus_overlap_areas.csv
